<a href="https://colab.research.google.com/github/Kangkrkr/my-colab/blob/master/0.%EC%BD%94%EB%9E%A9%EC%9D%84_%ED%99%9C%EC%9A%A9%ED%95%9C_sLM_%EB%9D%84%EC%9B%8C%EB%B3%B4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (269 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently i

In [2]:
!apt-get update -qq && apt-get install -y -qq pciutils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 122600 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Setting up pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.7.0-6) ...
Setting up pciutils (1:3.7.0-6) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbb.

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!OLLAMA_KEEP_ALIVE=-1 OLLAMA_FLASH_ATTENTION=1 nohup ollama serve > /dev/null 2>&1 &

In [5]:
!ollama pull gemma4:12b

In [4]:
!ollama pull bge-m3

In [13]:
!pip install -U \
  langchain langchain-core langgraph \
  langchain-classic langchain-community \
  langchain-ollama langchain-text-splitters \
  langchain-tavily langchain-neo4j langchain-redis \
  langchain-experimental rank_bm25 faiss-cpu scikit-learn networkx

  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.6.0
    Uninstalling hf-xet-1.6.0:
      Successfully uninstalled hf-xet-1.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires hf-xet<2.0.0,>=1.5.2; platform_machine == "x86_64" or platform_machine == "amd64" or platform_machine == "AMD64" or platform_machine == "arm64" or platform_machine == "aarch64", but you have hf-xet 1.2.0 which is incompatible.


In [7]:
import warnings, subprocess, time, requests
warnings.filterwarnings("ignore")
try:                                     # 런타임 재시작으로 서버가 죽었으면 다시 띄운다
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(10)

# ── 표 출력 유틸 (한글은 폭 2로 계산해야 정렬이 맞는다) ──────────────
import unicodedata

def _w(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in str(s))

def _clip(s, n):
    """표시 폭 기준으로 자른다. 말줄임은 폭이 환경마다 다른 '…' 대신 ASCII 를 쓴다."""
    s = str(s)
    if _w(s) <= n:
        return s
    out, w = "", 0
    for c in s:
        cw = 2 if unicodedata.east_asian_width(c) in "WF" else 1
        if w + cw > n - 2:          # ".." 자리 확보
            break
        out, w = out + c, w + cw
    return out + ".."

def _pad(s, n, right=False):
    gap = " " * max(0, n - _w(s))
    return (gap + str(s)) if right else (str(s) + gap)

def table(rows, headers, align=None, title=None, maxw=62):
    """모노스페이스 표를 그린다. align 은 각 열의 "<" 또는 ">"."""
    rows = [[_clip(c, maxw) for c in r] for r in rows]
    align = align or ["<"] * len(headers)
    w = [max([_w(headers[i])] + [_w(r[i]) for r in rows]) for i in range(len(headers))]
    rule = lambda l, m, r: l + m.join("─" * (x + 2) for x in w) + r
    if title:
        print(f"\n{title}")
    print(rule("┌", "┬", "┐"))
    print("│ " + " │ ".join(_pad(headers[i], w[i]) for i in range(len(headers))) + " │")
    print(rule("├", "┼", "┤"))
    for r in rows:
        print("│ " + " │ ".join(_pad(r[i], w[i], align[i] == ">")
                                for i in range(len(headers))) + " │")
    print(rule("└", "┴", "┘"))


def step(msg):
    """파이프라인의 흐름을 눈에 띄게 표시."""
    print(f"\n{'━' * 70}\n▶ {msg}\n{'━' * 70}")

# ── 실행 지표 수집: LLM 호출마다 지연시간·토큰을 기록한다 ────────────
import time
from langchain_core.callbacks import BaseCallbackHandler

class Trace(BaseCallbackHandler):
    def __init__(self):
        self.rows = []          # (호출번호, 지연초, 입력토큰, 출력토큰)
        self.t0 = time.time()
        self._t = None

    @property
    def n(self):
        return len(self.rows)

    def on_chat_model_start(self, *a, **k): self._t = time.time()
    def on_llm_start(self, *a, **k):        self._t = time.time()

    def on_llm_end(self, response, **k):
        dt = time.time() - (self._t or time.time())
        gen = response.generations[0][0]
        u = getattr(getattr(gen, "message", None), "usage_metadata", None) or {}
        self.rows.append((len(self.rows) + 1, dt,
                          u.get("input_tokens", 0), u.get("output_tokens", 0)))
        print(f"      · LLM #{len(self.rows):<3} {dt:6.1f}s")   # 진행 상황만 간단히

    def report(self):
        if not self.rows:
            return
        if len(self.rows) <= 12:
            table([[f"#{i}", f"{dt:.1f}s", f"{a:,}", f"{b:,}"] for i, dt, a, b in self.rows],
                  ["호출", "지연", "입력 토큰", "출력 토큰"],
                  [">", ">", ">", ">"], title="  LLM 호출 내역")
        el = time.time() - self.t0
        tin = sum(r[2] for r in self.rows)
        tout = sum(r[3] for r in self.rows)
        lat = [r[1] for r in self.rows]
        table([["LLM 호출", f"{len(self.rows)}회"],
               ["총 소요", f"{el:.1f}s"],
               ["평균 지연", f"{sum(lat) / len(lat):.1f}s"],
               ["최대 지연", f"{max(lat):.1f}s"],
               ["입력 토큰", f"{tin:,}"],
               ["출력 토큰", f"{tout:,}"],
               ["출력 속도", f"{tout / el:.1f} tok/s"]],
              ["지표", "값"], ["<", ">"], title="  실행 지표")

trace = Trace()

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

llm = ChatOllama(model="gemma4:12b", temperature=0, num_ctx=8192, reasoning=False, callbacks=[trace])
embeddings = OllamaEmbeddings(model="bge-m3")

TEXTS = [
    # ── 기술 (LangGraph / LangChain)
    "LangGraph는 StateGraph로 상태를 관리하며, 각 노드는 상태의 일부만 반환하면 자동으로 병합된다.",
    "LangGraph의 상태 스키마는 TypedDict로 정의하고, Annotated와 리듀서로 누적 방식을 지정한다.",
    "LangGraph 체크포인터는 스레드 단위로 상태 스냅샷을 저장해 중단 후 재개를 가능하게 한다.",
    "LangGraph의 조건부 엣지는 함수의 반환 문자열을 키로 삼아 다음 노드를 결정한다.",
    "LangGraph에서 interrupt 함수는 실행을 멈추고 사람의 입력을 기다리게 한다.",
    "LangChain LCEL은 파이프 연산자로 프롬프트, 모델, 파서를 이어 붙이는 선언적 문법이다.",
    "LangChain v1에서 레거시 체인과 리트리버는 langchain-classic 패키지로 이동했다.",
    # ── 인사 규정
    "회사 연차는 1년간 80퍼센트 이상 출근한 근로자에게 15일이 발생하며, 사용 시 팀장 승인이 필요하다.",
    "연차는 그룹웨어 휴가 메뉴에서 신청하며, 3일 이상 연속 사용 시 부서장 결재가 추가된다.",
    "재택근무는 주 2회까지 가능하며, 전날 오후 6시까지 그룹웨어에 신청해야 한다.",
    "재택근무 중에는 업무 시간 내 메신저 응답 의무가 있으며, 월 1회 사무실 출근일이 지정된다.",
    "출장비는 영수증 원본을 첨부해 귀사 후 5영업일 이내에 정산 신청해야 한다.",
    "시차출퇴근제는 오전 7시에서 10시 사이에 출근 시각을 선택할 수 있는 제도이다.",
    # ── 계약 / 법률
    "계약 해지는 당사자 일방이 30일 전 서면으로 통지함으로써 가능하다.",
    "계약 갱신은 만료 60일 전까지 갱신 의사를 통지하지 않으면 동일 조건으로 자동 연장된다.",
    "임대차 계약에서 보증금 반환은 목적물 인도와 동시이행 관계에 있다.",
    "전세 계약 만료 시 임차인이 갱신 거절을 통지하지 않으면 묵시적 갱신이 성립한다.",
    "이사 전에 관리비와 공과금을 정산하고 계량기 수치를 확인해야 보증금 반환이 지연되지 않는다.",
    "전입신고와 확정일자를 갖춘 임차인은 대항력과 우선변제권을 확보한다.",
    "임대인이 보증금을 돌려주지 않으면 임차권등기명령을 신청한 뒤 이사할 수 있다.",
    "손해배상액의 예정은 법원이 부당히 과다하다고 인정하면 감액할 수 있다.",
    # ── 기술 일반
    "RFC 7231에서 404 상태 코드는 서버가 요청한 표현을 찾지 못했음을 의미한다.",
    "HTTP 429 Too Many Requests는 클라이언트가 속도 제한을 초과했을 때 반환된다.",
    "JWT 토큰의 만료 시각은 exp 클레임에 유닉스 타임스탬프로 기록된다.",
]

raw_documents = [Document(page_content=t) for t in TEXTS]
vectorstore = FAISS.from_documents(raw_documents, embeddings)
print(f"인덱스 준비 완료: {vectorstore.index.ntotal}개 벡터 / {vectorstore.index.d}차원")

def show_docs(docs, title="검색 결과", scores=None):
    """검색 결과를 순위·점수와 함께 표로 표시."""
    has_score = scores is not None
    rows = []
    for i, d in enumerate(docs, 1):
        text = d if isinstance(d, str) else d.page_content
        rows.append([i] + ([f"{scores[i - 1]:.3f}"] if has_score else []) + [text])
    headers = ["#"] + (["점수"] if has_score else []) + ["문서"]
    align   = [">"] + ([">"]     if has_score else []) + ["<"]
    table(rows, headers, align, title=f"  {title} ({len(docs)}건)")

trace.t0 = time.time()   # 준비 과정은 제외하고 여기서부터 측정

# ────────────────────────────────────────────────────────────────────────

Q = "집 계약 끝날 때쯤 뭐 챙겨야 함?"   # 구어체·전문용어 없음 = HyDE 가 겨냥하는 질의

step("기준선: 질문을 그대로 임베딩해 검색")
base = vectorstore.similarity_search_with_score(Q, k=3)
show_docs([d for d, _ in base], "질문 임베딩", scores=[s for _, s in base])

step("HyDE 1단계: 가상 답변 문서 생성")
hypothetical = llm.invoke(
    "다음 질문에 대한 가상의 상세 답변 문단을 3문장으로 작성하세요. "
    f"사실 여부는 상관없습니다.\n질문: {Q}"
).content
print(hypothetical.strip())

step("HyDE 2단계: 가상 문서를 임베딩해 검색")
vec = embeddings.embed_query(hypothetical)
hyde = vectorstore.similarity_search_with_score_by_vector(vec, k=3)
show_docs([d for d, _ in hyde], "가상 문서 임베딩", scores=[s for _, s in hyde])

step("순위 비교 — 집합이 같아도 순위가 뒤집히면 답변 품질이 달라진다")
base_rank = {d.page_content: i + 1 for i, (d, _) in enumerate(base)}
rows = []
for i, (d, sc) in enumerate(hyde, 1):
    was = base_rank.get(d.page_content)
    rows.append([i, was if was else "-",
                 f"{was - i:+d}" if was else "신규",
                 f"{sc:.3f}", d.page_content])
table(rows, ["HyDE 순위", "기준선 순위", "변동", "거리", "문서"],
      [">", ">", ">", ">", "<"], title="  기준선 → HyDE 순위 변화")

# 가상 문서가 코퍼스에 없는 어휘를 만들어내면 검색이 그 방향으로 끌려갑니다(환각 전파).
# 어떤 단어가 다리를 놓았는지 보면 성공·실패 원인을 바로 알 수 있습니다.
step("가상 문서가 새로 끌어온 어휘")
import re as _re2

_PARTICLES = ("으로", "에서", "에게", "까지", "부터", "이나",
              "을", "를", "이", "가", "은", "는", "의", "에", "와", "과", "로", "도", "만")

def _stem(w):
    """조사만 떼어 낸 얕은 정규화 (형태소 분석기 없이 대략만)."""
    for p in sorted(_PARTICLES, key=len, reverse=True):
        if len(w) > len(p) + 1 and w.endswith(p):
            return w[: -len(p)]
    return w

_words = lambda t: {_stem(w) for w in _re2.findall(r"[가-힣]{2,}", t)}

hypo_words, q_words = _words(hypothetical), _words(Q)
rows = []
for d, _ in hyde[:3]:
    bridged = sorted((hypo_words - q_words) & _words(d.page_content))
    rows.append([d.page_content, ", ".join(bridged) if bridged else "-"])
table(rows, ["문서", "가상 문서와 겹치는 어휘"], ["<", "<"],
      title="  어휘 다리 (질문에는 없고 가상 문서에만 있던 단어)")
print("\n  → 겹치는 어휘가 질문의 의도와 맞으면 HyDE 가 이득이고,")
print("    엉뚱한 단어로 이어지면 오히려 순위가 나빠집니다. 이것이 '환각 전파'입니다.")

trace.report()

인덱스 준비 완료: 24개 벡터 / 1024차원

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 기준선: 질문을 그대로 임베딩해 검색
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  질문 임베딩 (3건)
┌───┬───────┬────────────────────────────────────────────────────────────────┐
│ # │ 점수  │ 문서                                                           │
├───┼───────┼────────────────────────────────────────────────────────────────┤
│ 1 │ 0.835 │ 이사 전에 관리비와 공과금을 정산하고 계량기 수치를 확인해야 .. │
│ 2 │ 0.939 │ 임대차 계약에서 보증금 반환은 목적물 인도와 동시이행 관계에 .. │
│ 3 │ 0.990 │ 임대인이 보증금을 돌려주지 않으면 임차권등기명령을 신청한 뒤.. │
└───┴───────┴────────────────────────────────────────────────────────────────┘

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ HyDE 1단계: 가상 답변 문서 생성
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
      · LLM #1      8.2s
이사 당일에는 가장 먼저 집 내부의 파손이나 오염 상태를 꼼꼼히 확인하여 원상복구 범위에 대한 분쟁을 방지해야 합니다. 또한, 관리비나 공과금 정산이 완료되었는지 확인하고 관리사무소에서 발행하는 정산 영수증을 반드시 챙겨야 합니다. 